# Exercise 02: Linear Regression – Medical Insurance Costs

**Dataset:** Individual medical costs billed by a US health insurer for 1'338 insured persons
(from Lantz, *Machine Learning with R*). Source: [Kaggle – Medical Cost Personal Dataset](https://www.kaggle.com/datasets/mirichoi0218/insurance).

| variable | meaning |
|---|---|
| `age` | age of the insured person (years) |
| `sex` | female / male |
| `bmi` | body mass index (kg/m²) |
| `children` | number of children covered by the insurance |
| `smoker` | yes / no |
| `region` | residential area in the US (northeast, northwest, southeast, southwest) |
| `charges` | medical costs billed by the insurance (USD per year) |

**Business question:** Which characteristics are associated with higher medical costs, and how well can the costs of a
new customer be predicted?

**Slides:** Part 03 «Regression models» – linear regression (model, OLS, R², RMSE, confidence vs. prediction interval,
assumptions and diagnostics, multiple predictors, maximum likelihood).

**After this exercise you can ...**
- compute an OLS line, R² and RMSE by hand,
- fit and interpret simple and multiple linear regression models (*ceteris paribus*),
- check the model assumptions with residual plots,
- explain why "linear" means linear in the parameters (interaction terms),
- distinguish confidence and prediction intervals.

## Concept

**Model.** Multiple linear regression describes the target as a weighted sum of the features plus an error term:

$$y = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p + \varepsilon$$

Each $\beta_j$ is the change in the *average* $y$ per unit of $x_j$, **holding the other features constant**
(*ceteris paribus*) – an association, not automatically a causal effect. "Linear" means linear in the parameters:
$x^2$, $\log x$ or products $x_1 \cdot x_2$ (interactions) are allowed.

**OLS.** Ordinary least squares chooses the $\beta$'s that minimise the residual sum of squares
$\text{SSE}=\sum_i (y_i-\hat y_i)^2$. With one predictor:

$$\hat\beta_1 = \frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sum_i (x_i-\bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x$$

**Goodness of fit.**

$$R^2 = 1-\frac{\text{SSE}}{\text{SST}}, \quad \text{SST}=\sum_i (y_i-\bar y)^2, \qquad
\text{RMSE} = \sqrt{\tfrac{1}{n}\,\text{SSE}}$$

$R^2$ = share of the variation of $y$ explained by the model; RMSE = typical prediction error in the units of $y$.

**Confidence vs. prediction interval.** The 95 % *confidence interval* covers the **average** $y$ of all persons with the
given $x$; the 95 % *prediction interval* covers the $y$ of **one new** person and is always wider.

**Assumptions** (check with residual plots): linearity, independence, constant variance (homoscedasticity), normality of
the errors (for exact small-sample inference), no perfect multicollinearity (check with the VIF).

**Maximum likelihood.** Choose the parameters under which the observed data are most plausible. For independent normal
errors with constant variance, ML gives exactly the OLS solution.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: OLS by hand

Five insured persons (costs in 1'000 USD):

| age $x$ | 20 | 30 | 40 | 50 | 60 |
|---|---|---|---|---|---|
| costs $y$ | 2 | 5 | 6 | 9 | 13 |

a) Compute $\bar x$, $\bar y$, the slope $\hat\beta_1$ and the intercept $\hat\beta_0$ **by hand**.

b) Compute the fitted values, the residuals, SSE, SST, $R^2$ and the RMSE.

c) Interpret $\hat\beta_1$ and $\hat\beta_0$. What are the predicted costs for a 45-year-old person?

d) Verify your results with `numpy`.

In [ ]:
x = np.array([20, 30, 40, 50, 60])
y = np.array([2, 5, 6, 9, 13])

beta_1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
beta_0 = y.mean() - beta_1 * x.mean()
y_hat  = beta_0 + beta_1 * x
res    = y - y_hat
sse    = np.sum(res**2)
sst    = np.sum((y - y.mean())**2)

print('x_mean =', x.mean(), ' y_mean =', y.mean())
print('beta_1 =', round(beta_1, 3), ' beta_0 =', round(beta_0, 3))
print('fitted values:', y_hat.round(2))
print('residuals:    ', res.round(2))
print('SSE =', round(sse, 2), ' SST =', round(sst, 2))
print('R2 =', round(1 - sse/sst, 4), ' RMSE =', round(np.sqrt(sse/len(y)), 3))
print('Prediction for age 45:', round(beta_0 + beta_1 * 45, 2))

# Check with numpy's polyfit
print('np.polyfit:', np.polyfit(x, y, deg=1).round(3))

**Solution:**

a) $\bar x = 40$, $\bar y = 7$. $\sum(x_i-\bar x)(y_i-\bar y) = 100+20+0+20+120 = 260$,
$\sum(x_i-\bar x)^2 = 400+100+0+100+400 = 1000$ → $\hat\beta_1 = 0.26$, $\hat\beta_0 = 7-0.26\cdot 40 = -3.4$.

b) Fitted values 1.8, 4.4, 7.0, 9.6, 12.2; residuals 0.2, 0.6, −1.0, −0.6, 0.8;
SSE = 0.04+0.36+1+0.36+0.64 = 2.4; SST = 25+4+1+4+36 = 70; $R^2 = 1-2.4/70 \approx 0.966$;
RMSE $=\sqrt{2.4/5} \approx 0.69$ (i.e. about 690 USD).

c) Per additional year of age, costs are on average 260 USD higher. The intercept (−3'400 USD at age 0) lies far outside
the data and has no substantive meaning. Prediction for age 45: $-3.4 + 0.26 \cdot 45 = 8.3$ → 8'300 USD.

## Exercise 2: Import and explore the data

a) Import the data, check for missing values and duplicates (remove duplicates).

b) Plot a histogram of `charges`. Describe the distribution.

c) Create a scatter plot of `charges` against `age`, coloured by `smoker`, and box plots of `charges` by `smoker`.
What do you see?

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/insurance.csv', sep=',')

# Missing values and duplicates
print('Missing values:', df.isna().sum().sum())
print('Duplicates:    ', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df.describe().round(1))

# b) Histogram of charges
plt.figure(figsize=(7, 3.5))
plt.hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Charges [USD per year]')
plt.ylabel('Number of persons')
plt.title('Distribution of medical charges')
plt.show()

# c) Scatter plot and box plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={'width_ratios': [2, 1]})
sns.scatterplot(data=df, x='age', y='charges', hue='smoker', s=15, ax=axes[0])
axes[0].set_title('Charges vs. age')
sns.boxplot(data=df, x='smoker', y='charges', ax=axes[1])
axes[1].set_title('Charges by smoking status')
plt.show()

**Solution:**

a) No missing values, one duplicated row (removed) → 1'337 persons.

b) The distribution is strongly **right-skewed**: most persons have costs below 15'000 USD, a smaller group has
costs between 30'000 and 60'000 USD.

c) Costs rise with age, but the scatter plot shows **three bands** that run in parallel. The upper band consists
exclusively of smokers, the lowest band almost exclusively of non-smokers. Smokers have much higher costs (median about
35'000 USD vs. about 7'000 USD). Smoking status is obviously a key predictor – a model with age alone will miss this.

## Exercise 3: Simple linear regression

a) Split the data into a training set (80 %) and a test set (20 %) with `random_state=42`
(split the whole data frame: `train_df, test_df = train_test_split(df, ...)`).

b) Fit the model `charges ~ age` on the training set with `smf.ols()` and print the summary.

c) Interpret the slope and $R^2$. Plot the regression line into the scatter plot of the training data.

In [ ]:
# a) Train/test split of the data frame
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)
print('Training set:', train_df.shape, ' Test set:', test_df.shape)

# b) Simple linear regression
model_1 = smf.ols('charges ~ age', data=train_df).fit()
print(model_1.summary())

# c) Scatter plot with regression line
age_grid = pd.DataFrame({'age': np.arange(18, 65)})
plt.figure(figsize=(7, 4))
plt.scatter(train_df['age'], train_df['charges'], s=10, alpha=0.5)
plt.plot(age_grid['age'], model_1.predict(age_grid), color='darkred', linewidth=2, label='OLS line')
plt.xlabel('Age [years]')
plt.ylabel('Charges [USD]')
plt.legend()
plt.show()

**Solution:**

Per additional year of age the expected costs increase by about **240 USD** (p < 0.001). However, $R^2$ is only about
**0.08**: age explains less than 10 % of the variation of the costs. The line runs between the three bands and fits none
of them well.

## Exercise 4: Residual diagnostics

a) Plot the residuals of `model_1` against the fitted values, colouring the points by `smoker`.

b) Create a Q-Q plot of the residuals (`sm.qqplot(..., line='45', fit=True)`).

c) Which assumptions are violated? What is the most likely reason?

In [ ]:
# a) Residuals vs. fitted values
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(x=model_1.fittedvalues, y=model_1.resid, hue=train_df['smoker'], s=12, ax=axes[0])
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs. fitted (model 1)')

# b) Q-Q plot
sm.qqplot(model_1.resid, line='45', fit=True, ax=axes[1])
axes[1].set_title('Q-Q plot of residuals (model 1)')
plt.show()

**Solution:**

The residuals are not randomly scattered around zero: they form bands, and almost all large positive residuals belong to
smokers. → **Linearity / correct specification is violated** (an important predictor is missing), and the spread is not
constant. The Q-Q plot shows a strong deviation from the line in the upper tail → residuals are **not normally
distributed** (right-skewed). The most likely reason is the omitted variable `smoker` (and possibly `bmi`).

## Exercise 5: Multiple linear regression

a) Fit the model `charges ~ age + bmi + children + smoker + sex + region` on the training set. `statsmodels` creates the
dummy variables for the categorical features automatically.

b) Interpret the coefficients of `smoker`, `age` and `bmi` (*ceteris paribus*). Which predictors are not significant at
the 5 % level?

c) How did $R^2$ change compared to `model_1`?

In [ ]:
# a) Multiple linear regression
model_2 = smf.ols('charges ~ age + bmi + children + smoker + sex + region', data=train_df).fit()
print(model_2.summary())

**Solution:**

- **smoker[T.yes]** ≈ +23'100 USD: a smoker has on average about 23'100 USD higher costs than a non-smoker *of the same
  age, BMI, number of children, sex and region*.
- **age** ≈ +250 USD per year, holding the other predictors constant.
- **bmi** ≈ +320 USD per BMI unit, holding the other predictors constant.
- `sex` and the region dummies are not significant at the 5 % level (p > 0.05) – they add little explanatory power.

$R^2$ rises from about 0.08 to about **0.73**. (Remember: an association, not automatically a causal effect.)

## Exercise 6: Interaction – "linear" means linear in the parameters

Does a high BMI have the same effect for smokers and non-smokers? An **interaction term** lets the slope of `bmi` differ
between the two groups:

$$\text{charges} = \beta_0 + \beta_1\,\text{age} + \beta_2\,\text{bmi} + \beta_3\,\text{smoker} + \beta_4\,(\text{bmi}\cdot\text{smoker}) + \beta_5\,\text{children} + \varepsilon$$

a) First, create a scatter plot of `charges` against `bmi`, coloured by `smoker`.

b) Fit the model `charges ~ age + children + bmi * smoker` (the `*` adds both main effects and the interaction).

c) What is the estimated effect of +1 BMI unit for a non-smoker and for a smoker? How did $R^2$ change?

d) Repeat the residual plot of Exercise 4 for this model. What has improved?

In [ ]:
# a) Scatter plot charges vs. bmi
plt.figure(figsize=(7, 4))
sns.scatterplot(data=train_df, x='bmi', y='charges', hue='smoker', s=12)
plt.title('Charges vs. BMI')
plt.show()

# b) Model with interaction
model_3 = smf.ols('charges ~ age + children + bmi * smoker', data=train_df).fit()
print(model_3.summary())

# c) Effect of +1 BMI for non-smokers and smokers
p = model_3.params
print('Effect of +1 BMI (non-smoker):', round(p['bmi']))
print('Effect of +1 BMI (smoker):    ', round(p['bmi'] + p['bmi:smoker[T.yes]']))
print('R2 model 2:', round(model_2.rsquared, 3), ' R2 model 3:', round(model_3.rsquared, 3))

# d) Residual plot
plt.figure(figsize=(7, 4))
sns.scatterplot(x=model_3.fittedvalues, y=model_3.resid, hue=train_df['smoker'], s=12)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs. fitted (model 3)')
plt.show()

**Solution:**

a) For non-smokers the costs hardly depend on BMI. For smokers the costs rise steeply with BMI – smokers with a BMI
above about 30 (obese) form the top cost group.

c) For non-smokers +1 BMI changes the costs by only a few USD (not significant); for smokers +1 BMI increases the costs by
about **1'460 USD**. The effect of BMI therefore depends on smoking status. $R^2$ increases from about 0.73 to about
**0.82** – although the model is still *linear in the parameters*.

d) The residuals of smokers are now centred around zero; most of the band structure has disappeared. A group of persons
with large positive residuals remains (costs that are not explained by the predictors), so the model is better but still
not perfect.

## Exercise 7: Evaluation on the test set

So far $R^2$ was computed on the training data. Compute **RMSE** and **$R^2$ on the test set** for `model_1`, `model_2`
and `model_3` and summarise the results in a table. Which model do you recommend?

In [ ]:
results = []
for name, m in [('model 1: age', model_1),
                ('model 2: all main effects', model_2),
                ('model 3: with bmi x smoker', model_3)]:
    y_pred = m.predict(test_df)
    results.append({'model': name,
                    'R2 (train)': round(m.rsquared, 3),
                    'R2 (test)': round(r2_score(test_df['charges'], y_pred), 3),
                    'RMSE (test)': round(root_mean_squared_error(test_df['charges'], y_pred))})
print(pd.DataFrame(results).to_string(index=False))

**Solution:**

Model 3 has the highest test $R^2$ (≈ 0.88) and the smallest test RMSE (≈ 4'600 USD) compared to ≈ 6'000 USD for
model 2 and ≈ 12'900 USD for model 1. The test $R^2$ is not lower than the training $R^2$ (here even slightly higher by
chance), so there is no sign of overfitting – not surprising for models with only a handful of parameters and > 1'000
observations.
Model 3 is recommended: it is more accurate and still easy to interpret.

## Exercise 8: Confidence vs. prediction interval

A new customer is 45 years old, smokes, has a BMI of 32 and two children.

a) Compute the predicted costs with `model_3` and the 95 % confidence and prediction intervals
(`model_3.get_prediction(new_customer).summary_frame(alpha=0.05)`).

b) Interpret both intervals. Which one is relevant if the insurer wants to know the costs of **this** customer, and which
one if it wants to know the **average** costs of all such customers?

In [ ]:
new_customer = pd.DataFrame({'age': [45], 'children': [2], 'bmi': [32.0], 'smoker': ['yes']})

pred = model_3.get_prediction(new_customer).summary_frame(alpha=0.05)
print(pred.round(0).to_string())
print(f"""
Predicted costs:            {pred['mean'].iloc[0]:,.0f} USD
95% confidence interval:    {pred['mean_ci_lower'].iloc[0]:,.0f} - {pred['mean_ci_upper'].iloc[0]:,.0f} USD
95% prediction interval:    {pred['obs_ci_lower'].iloc[0]:,.0f} - {pred['obs_ci_upper'].iloc[0]:,.0f} USD""")

**Solution:**

The model predicts about 35'900 USD per year. The **confidence interval** (≈ 35'200–36'700 USD) tells us where the
*average* costs of all 45-year-old smokers with BMI 32 and two children lie – relevant e.g. for pricing a tariff for this
group. The **prediction interval** (≈ 26'200–45'600 USD) tells us where the costs of *this one* customer will lie with
95 % probability – relevant for the individual risk. It is much wider because it also contains the scatter of individual
persons around the regression line.

## Exercise 9: Multicollinearity (VIF)

Compute the variance inflation factor (VIF) for the numerical predictors `age`, `bmi`, `children` of the training set
(add a constant column with `sm.add_constant()` first). Is multicollinearity a problem here? What would happen if you
added the variable `age_in_months = 12 * age` to the model?

In [ ]:
X_vif = sm.add_constant(train_df[['age', 'bmi', 'children']])

vif = pd.DataFrame({'feature': X_vif.columns,
                    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
print(vif[vif['feature'] != 'const'].round(2).to_string(index=False))

**Solution:**

All VIF values are close to 1 → the predictors are almost uncorrelated, multicollinearity is not a problem.
`age_in_months` would be a perfect linear function of `age` (**perfect multicollinearity**): the coefficients would not be
identifiable (OLS cannot decide how to split the effect between the two variables; the VIF would be infinite).

## Exercise 10 (optional): Maximum likelihood

The share of smokers in the data is to be estimated. Suppose in a random sample of $n=10$ insured persons, $k=2$ smoke.
For a smoking probability $p$ the likelihood of this sample (in a given order) is $L(p) = p^{k}(1-p)^{n-k}$.

a) Compute $L(p)$ for $p = 0.1, 0.2, 0.5$ by hand.

b) Plot $L(p)$ for $p \in [0, 1]$ and find the maximum. Compare with the share of smokers in the full data set.

In [ ]:
n, k = 10, 2

def likelihood(p):
    return p**k * (1 - p)**(n - k)

for p in [0.1, 0.2, 0.5]:
    print(f'L({p}) = {likelihood(p):.5f}')

p_grid = np.linspace(0, 1, 501)
p_hat = p_grid[np.argmax(likelihood(p_grid))]

plt.figure(figsize=(6, 3.5))
plt.plot(p_grid, likelihood(p_grid))
plt.axvline(p_hat, color='darkred', linestyle='--', label=f'maximum at p = {p_hat:.2f}')
plt.xlabel('p')
plt.ylabel('L(p)')
plt.title('Likelihood of 2 smokers in 10 persons')
plt.legend()
plt.show()

print('Share of smokers in the full data set:', round((df['smoker'] == 'yes').mean(), 3))

**Solution:**

a) $L(0.1) = 0.1^2 \cdot 0.9^8 \approx 0.00430$, $L(0.2) = 0.2^2 \cdot 0.8^8 \approx 0.00671$,
$L(0.5) = 0.5^{10} \approx 0.00098$.

b) The maximum lies at $\hat p = k/n = 0.2$ – the ML estimate is simply the observed share. In the full data set about
20.5 % of the persons smoke. The same principle (choose the parameters that make the data most plausible) is used to fit
logistic regression (Exercise 04).

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')